### RAG Pipeline

Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [9]:
# Read all PDF files in directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files inside a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Finding all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Adding source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal Documents Loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process.

Processing: agriculture.pdf
Loaded 7 pages

Processing: attention.pdf
Loaded 11 pages

Total Documents Loaded: 18


In [11]:
# Splitting text and Chunking them together

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Spliting {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [12]:
chunks = split_documents(all_pdf_documents)

chunks

Spliting 18 documents into 91 chunks

Example Chunk:
Content: GENERAL ARTICLES 
 
CURRENT SCIENCE, VOL. 105, NO. 5, 10 SEPTEMBER 2013  587
History of agricultural research in India 
 
Anwesha Borthakur* and Pardeep Singh  
 
India as a predominantly agricultural...
Metadata: {'producer': 'Acrobat Distiller 7.0 (Windows)', 'creator': 'Acrobat PDFMaker 7.0 for Word', 'creationdate': '2013-09-11T20:00:40+05:30', 'author': 'system', 'moddate': '2013-09-11T20:00:44+05:30', 'sourcemodified': 'D:20130909083917', 'title': 'On the origin of the artesian', 'source': '..\\data\\pdf_files\\agriculture.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'agriculture.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 7.0 (Windows)', 'creator': 'Acrobat PDFMaker 7.0 for Word', 'creationdate': '2013-09-11T20:00:40+05:30', 'author': 'system', 'moddate': '2013-09-11T20:00:44+05:30', 'sourcemodified': 'D:20130909083917', 'title': 'On the origin of the artesian', 'source': '..\\data\\pdf_files\\agriculture.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'agriculture.pdf', 'file_type': 'pdf'}, page_content='GENERAL ARTICLES \n \nCURRENT SCIENCE, VOL. 105, NO. 5, 10 SEPTEMBER 2013  587\nHistory of agricultural research in India \n \nAnwesha Borthakur* and Pardeep Singh  \n \nIndia as a predominantly agricultural country attributes a major share of its overall development to \nthe agriculture sector. Indian agric ulture is a miscellane ous and extensive sector involving a large \nnumber of stakeholders. India has one of the largest and institutionally most complex agricultural \nresearch systems in the world. Ho wever, such a complex re

In [14]:
# Embedding and VectorStoreDB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding genertion using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the Embedding Manager

        Args: model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer Model"""

        try:
            print(f"Loading the Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding Model Loaded Successfully!")
            print(f"Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate Embeddings for a list of texts

        Args: texts: List of text strings to embed

        Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model Not Found")

        print(f"Generating Embeddings for {len(texts)} texts...")
        
        embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"Embeddings Generated Successfully")
        print(f"Embeddings Shape: {embeddings.shape}")

        return embeddings

In [18]:
# Initialize the Embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading the Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8051.54it/s]


Embedding Model Loaded Successfully!
Embedding Dimension: 384


In [ ]:
# VectorStore

class VectorStore:
    """Managing document embeddings in a ChromaDB Vector Store"""